# 03 - 搜索、消息、用户测试

测试内容:
1. 搜索文档 (GET /drive/v1/files)
2. 发送消息 (POST /im/v1/messages)
3. 搜索用户 (POST /contact/v3/users/batch_get_id)
4. 获取群聊列表 (GET /im/v1/chats)

参考文档:
- 搜索: https://open.feishu.cn/document/server-docs/docs/drive-v1/file/search
- 消息: https://open.feishu.cn/document/server-docs/docs/im-v1/message/create
- 用户: https://open.feishu.cn/document/server-docs/docs/contact-v3/user/batch_get_id

In [ ]:
import os
import json
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../.env'))
except ImportError:
    pass

from feishu_client import FeishuClient

client = FeishuClient()
print("✓ 客户端初始化成功")

## 3.1 搜索文档

In [ ]:
# 搜索云空间中的文档
SEARCH_KEYWORD = "测试"  # 修改为你的搜索词

try:
    result = client.api(
        'POST',
        '/drive/v1/files/search',
        json_data={'search_key': SEARCH_KEYWORD, 'count': 20}
    )
    
    files = result.get('files', [])
    print(f"搜索 '{SEARCH_KEYWORD}' 找到 {len(files)} 个结果\n")
    
    for i, f in enumerate(files[:10]):
        print(f"{i+1}. {f.get('name', '未命名')} ({f.get('type', 'unknown')})")
        print(f"   URL: {f.get('url', 'N/A')}")
        print(f"   Token: {f.get('token', 'N/A')}")
        print()
except Exception as e:
    print(f"搜索失败: {e}")

## 3.2 列出云空间文件

In [ ]:
# 列出最近编辑的文件
try:
    result = client.api(
        'GET',
        '/drive/v1/files',
        params={'page_size': 20, 'order_by': 'EditedTime', 'direction': 'DESC'}
    )
    
    files = result.get('files', [])
    print(f"最近编辑的 {len(files)} 个文件:\n")
    
    for i, f in enumerate(files):
        print(f"{i+1}. {f.get('name', '未命名')} ({f.get('type', 'unknown')})")
        print(f"   Token: {f.get('token', 'N/A')}")
        print()
except Exception as e:
    print(f"列出失败: {e}")

## 3.3 发送消息

In [ ]:
# ⚠ 发送消息前需要知道接收者的 open_id 或 chat_id
# 可以通过 3.4 获取群聊列表，或 3.5 查找用户

# 示例：发送给机器人自己（测试用）
# 先获取机器人信息
try:
    bot_info = client.api('GET', '/bot/v3/info')
    bot_open_id = bot_info.get('open_id')
    bot_name = bot_info.get('app_name', 'Unknown')
    print(f"机器人: {bot_name}")
    print(f"open_id: {bot_open_id}")
except Exception as e:
    print(f"获取机器人信息失败: {e}")
    bot_open_id = None

In [ ]:
# 发送文本消息
if bot_open_id:
    try:
        result = client.api(
            'POST',
            '/im/v1/messages',
            json_data={
                'receive_id': bot_open_id,
                'msg_type': 'text',
                'content': json.dumps({'text': 'Hello from Python API! 👋\n这是通过飞书 Open API 发送的测试消息。'})
            },
            params={'receive_id_type': 'open_id'}
        )
        print(f"✓ 消息发送成功! message_id: {result.get('message_id')}")
    except Exception as e:
        print(f"发送失败: {e}")
else:
    print("未获取到 open_id，跳过发送")

## 3.4 获取群聊列表

In [ ]:
try:
    result = client.api('GET', '/im/v1/chats', params={'page_size': 50})
    chats = result.get('items', [])
    print(f"共 {len(chats)} 个群聊:\n")
    
    for i, c in enumerate(chats[:10]):
        print(f"{i+1}. {c.get('name', '未命名')} ({c.get('chat_type', 'unknown')})")
        print(f"   chat_id: {c.get('chat_id')}")
        print(f"   成员数: {c.get('member_count', 'N/A')}")
        print()
except Exception as e:
    print(f"获取群聊失败: {e}")

## 3.5 查找用户

In [ ]:
# 通过邮箱查找用户 open_id
TEST_EMAIL = os.environ.get('FEISHU_TEST_EMAIL', '')  # 设置测试邮箱

if TEST_EMAIL:
    try:
        result = client.api(
            'POST',
            '/contact/v3/users/batch_get_id',
            json_data={'emails': [TEST_EMAIL]},
            params={'user_id_type': 'open_id'}
        )
        users = result.get('user_list', [])
        if users:
            user = users[0]
            print(f"用户: {user.get('name', 'N/A')}")
            print(f"open_id: {user.get('user_id') or user.get('open_id')}")
            print(f"邮箱: {TEST_EMAIL}")
        else:
            print("未找到用户")
    except Exception as e:
        print(f"查找失败: {e}")
else:
    print("未设置 FEISHU_TEST_EMAIL，跳过用户查找")
    print("提示: 在 .env 中添加 FEISHU_TEST_EMAIL=your@email.com")

## 3.6 获取应用信息

In [ ]:
try:
    result = client.api('GET', '/bot/v3/info')
    print(json.dumps(result, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"获取应用信息失败: {e}")